# Linear stability: the Orr–Sommerfeld eigenvalue problem

**Book:** §6.1.2, Figure 6.3 &nbsp;·&nbsp; `ch06/orr_sommerfeld.ipynb`

The stability of plane Poiseuille flow, U(y)=1−y², is governed by the **Orr–Sommerfeld equation**
for the perturbation streamfunction φ(y):

$$(U-c)(\varphi''-\alpha^2\varphi)-U''\varphi=\frac{1}{i\alpha Re}\left(\varphi''''-2\alpha^2\varphi''+\alpha^4\varphi\right),\qquad \varphi(\pm1)=\varphi'(\pm1)=0.$$

Here **c = c_r + i c_i is a complex eigenvalue** (the phase speed); c_i > 0 means the mode grows —
the flow is unstable. The canonical Orszag (1971) benchmark: at Re=10000, α=1 the least-stable mode
is **c = 0.23753 + 0.00374 i** (unstable — this is the Tollmien–Schlichting wave).

**A PINN as an eigenvalue solver.** The network outputs (φ_r, φ_i); clamped BCs are hard via the
trial φ=(1−y²)²·N(y); the eigenvalue **c is two trainable scalars**; and a normalization penalty
∫|φ|²=1 prevents the trivial φ≡0. There is no data — the residual is the whole loss.

**The honest finding.** An eigenvalue problem has *many* eigenvalues, and which one the PINN finds
depends on the initialisation — a live instance of §1.5.3 (a non-convex loss has many minima). Of
three initialisations, one **locks onto the Orszag eigenvalue** (residual → 3×10⁻⁴, c_i > 0,
correctly unstable); the others settle on spurious modes (residual plateaus ~10⁻²). **The residual
history is the diagnostic** — it cleanly separates the converged run from the plateaued ones.

*(The Squire equation for the normal vorticity decouples for 2-D perturbations, and Squire's
theorem guarantees those modes are stable, so Orr–Sommerfeld is the critical one.)*

Warning: three eigenvalue searches with fourth-order autograd — the slowest example in the kit.
Use a GPU.

In [ ]:
"""Orr-Sommerfeld eigenvalue problem for plane Poiseuille flow, solved as a PINN.
Base flow U(y)=1-y^2 on y in [-1,1].  OS equation for the perturbation streamfunction phi(y):

  (U-c)(phi'' - a^2 phi) - U'' phi  -  (1/(i a Re)) (phi'''' - 2 a^2 phi'' + a^4 phi) = 0
  phi(+-1)=phi'(+-1)=0.

c = c_r + i c_i is the complex eigenvalue (phase speed); c_i>0 => unstable.
Benchmark (Orszag 1971): Re=10000, a=1  ->  c = 0.23752649 + 0.00373967 i  (least stable mode).

PINN:  network y->(phi_r,phi_i);  hard clamped BCs via the trial phi=(1-y^2)^2 * N(y);
c = two trainable scalars;  a normalization penalty (int|phi|^2 = 1) prevents the trivial phi=0.
The eigenvalue found depends on the initialization -- an eigenvalue problem has many eigenvalues,
and gradient descent finds whichever basin it starts in.
"""
import time, json, numpy as np, torch, torch.nn as nn, matplotlib
import matplotlib.pyplot as plt

dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", dev)
ALPHA, RE = 1.0, 10000.0
NU = 1.0/(ALPHA*RE)                          # 1/(a Re) = 1e-4;  1/(i a Re) = -i NU
C_REF = complex(0.23752649, 0.00373967)      # Orszag 1971 benchmark
def g1(f,x): return torch.autograd.grad(f,x,torch.ones_like(f),create_graph=True)[0]

def build():
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(1,64),nn.Tanh(),nn.Linear(64,64),nn.Tanh(),
                         nn.Linear(64,64),nn.Tanh(),nn.Linear(64,2)).to(dev)
def phi(net,y):                              # hard phi=phi'=0 at y=+-1 via (1-y^2)^2
    o=net(y); w=(1-y**2)**2
    return w*o[:,0:1], w*o[:,1:2]

def train(cr0, ci0, epochs=20000, wn=10.0):
    net=build(); c=nn.Parameter(torch.tensor([cr0,ci0],dtype=torch.float32,device=dev))
    opt=torch.optim.Adam(list(net.parameters())+[c],2e-3)
    yg=torch.linspace(-1,1,400,device=dev).reshape(-1,1)
    res_hist=[]; c_hist=[]; t0=time.perf_counter()
    for e in range(epochs):
        if e==int(.6*epochs):
            for gp in opt.param_groups: gp['lr']=4e-4
        if e==int(.85*epochs):
            for gp in opt.param_groups: gp['lr']=8e-5
        opt.zero_grad()
        y=(torch.rand(600,1,device=dev)*2-1).requires_grad_(True)
        pr,pi=phi(net,y)
        pr1=g1(pr,y);pr2=g1(pr1,y);pr3=g1(pr2,y);pr4=g1(pr3,y)
        pi1=g1(pi,y);pi2=g1(pi1,y);pi3=g1(pi2,y);pi4=g1(pi3,y)
        U=1-y**2; Upp=-2.0; a2=ALPHA**2; a4=ALPHA**4
        cr,ci=c[0],c[1]
        L2r=pr2-a2*pr; L2i=pi2-a2*pi
        Vr=pr4-2*a2*pr2+a4*pr; Vi=pi4-2*a2*pi2+a4*pi
        rr=(U-cr)*L2r+ci*L2i-Upp*pr-NU*Vi
        ri=(U-cr)*L2i-ci*L2r-Upp*pi+NU*Vr
        loss_res=(rr**2+ri**2).mean()
        prg,pig=phi(net,yg); integ=torch.trapz((prg**2+pig**2).ravel(),yg.ravel())
        (loss_res+wn*(integ-1.0)**2).backward(); opt.step()
        if e%50==0:
            res_hist.append(loss_res.item()); c_hist.append([c[0].item(),c[1].item()])
        if e%4000==0:
            print(f"   e={e:5d}  c=({c[0].item():+.5f},{c[1].item():+.5f})  res={loss_res.item():.1e}")
    if dev.type=='cuda': torch.cuda.synchronize()
    with torch.no_grad(): prg,pig=phi(net,yg)
    return dict(cr=c[0].item(), ci=c[1].item(), res_hist=res_hist, c_hist=c_hist,
                yy=yg.cpu().numpy().ravel(), pr=prg.cpu().numpy().ravel(),
                pi=pig.cpu().numpy().ravel(), t=time.perf_counter()-t0)

print(f"reference (Orszag): c = {C_REF.real:.6f} + {C_REF.imag:.6f} i\n")
INITS={'A: $c_0=0.24+0.005i$':(0.24,0.005),
       'B: $c_0=0.20+0.000i$':(0.20,0.0),
       'C: $c_0=0.30-0.020i$':(0.30,-0.02)}
runs={}
for tag,(cr0,ci0) in INITS.items():
    d=train(cr0,ci0); err=abs(complex(d['cr'],d['ci'])-C_REF); d['err']=err
    runs[tag]=d
    print(f"{tag}: c=({d['cr']:+.5f},{d['ci']:+.5f})  |c-c_ref|={err:.2e}  "
          f"res_final={d['res_hist'][-1]:.1e}  [{d['t']:.0f}s]\n")

json.dump({t:{k:runs[t][k] for k in('cr','ci','err','t')} for t in runs},
          open("os_metrics.json","w"),indent=1)

# ------------------------------------------------------------- 3-panel story figure
best=min(runs,key=lambda k:runs[k]['err'])
fig,ax=plt.subplots(1,3,figsize=(15,4.4))
cols={'A: $c_0=0.24+0.005i$':'tab:red','B: $c_0=0.20+0.000i$':'tab:green','C: $c_0=0.30-0.020i$':'tab:orange'}
# (a) residual history -- convergence vs plateau
for t,d in runs.items():
    ax[0].semilogy(np.arange(len(d['res_hist']))*50, d['res_hist'], color=cols[t], lw=1.4, label=t)
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('mean squared residual')
ax[0].grid(alpha=.3,which='both'); ax[0].legend(fontsize=8)
ax[0].set_title('(a) Residual: one init converges, others plateau')
# (b) eigenvalue trajectory in the complex plane
ax[1].plot(C_REF.real,C_REF.imag,'k*',ms=18,mec='k',mfc='gold',zorder=6,label='Orszag benchmark')
for t,d in runs.items():
    ch=np.array(d['c_hist'])
    ax[1].plot(ch[:,0],ch[:,1],color=cols[t],lw=1.2,alpha=.8)
    ax[1].plot(ch[-1,0],ch[-1,1],'o',color=cols[t],ms=8,mec='k',mew=.5)
ax[1].axhline(0,color='gray',lw=.8,ls=':'); ax[1].set_xlabel('$c_r$'); ax[1].set_ylabel('$c_i$')
ax[1].grid(alpha=.3); ax[1].legend(fontsize=8)
ax[1].set_title('(b) Eigenvalue trajectory ($c_i>0$: unstable)')
# (c) the recovered eigenfunction (best init)
d=runs[best]; amp=np.sqrt(d['pr']**2+d['pi']**2); amp=amp/amp.max()
ax[2].plot(amp,d['yy'],'b',lw=2.2,label='$|\\phi|$')
ax[2].fill_betweenx(d['yy'],0,amp,color='tab:blue',alpha=.12)
ax[2].set_xlabel('$|\\phi(y)|$  (normalised)'); ax[2].set_ylabel('y')
ax[2].grid(alpha=.3); ax[2].legend(fontsize=9)
ax[2].set_title(f'(c) TS eigenmode  $c$=({d["cr"]:.4f},{d["ci"]:+.4f})')
plt.tight_layout(); plt.show()
print("METRICS",json.dumps({t:{'cr':runs[t]['cr'],'ci':runs[t]['ci'],'err':runs[t]['err']} for t in runs},indent=1))
